In [0]:
import json

def get_notebook_run_links():
    try:
        try:
            from dbruntime.databricks_repl_context import get_context
            ctx = get_context()
            
            # Extract basic host metadata directly
            browser_host = getattr(ctx, "browserHostName", None)
            workspace_id = getattr(ctx, "workspaceId", "N/A")
            
            # Read tags dictionary
            tags = getattr(ctx, "isInJob", {}) # falls back gracefully
            if hasattr(ctx, "get_tags"):
                tags = ctx.get_tags()
            elif hasattr(ctx, "__dict__"):
                tags = ctx.__dict__
        
        # Fallback to safe internal API if REPL context import isn't present
        except ImportError:
            # Use safeToJson() instead of toJson() to bypass Unity Catalog restrictions
            context_str = dbutils.notebook.entry_point.getDbutils().notebook().getContext().safeToJson()
            context = json.loads(context_str)
            tags = context.get("attributes", {}) or context.get("tags", {})
            browser_host = tags.get("browserHostName")
            workspace_id = tags.get("orgId", "N/A")

        # 1. Base URL formulation
        if browser_host:
            base_url = f"https://{browser_host}"
        else:
            try:
                # Direct lookup via Spark config
                base_url = spark.conf.get("spark.databricks.workspaceUrl")
                if not base_url.startswith("http"):
                    base_url = f"https://{base_url}"
                browser_host = base_url.replace("https://", "")
            except Exception:
                workspace_url = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().get()
                base_url = workspace_url.replace("api/2.0", "").rstrip("/")
                browser_host = base_url.replace("https://", "")

        # 2. Extract job-specific parameters
        job_name = tags.get("jobName", "Interactive Notebook")
        job_id = tags.get("jobId")
        
        run_id_obj = tags.get("currentRunId", {})
        run_id = run_id_obj.get("id") if isinstance(run_id_obj, dict) else tags.get("runId")
        job_run_id = tags.get("jobRunId", run_id)
        
        user = tags.get("user", "System")
        launched_by = "Manually" if user != "System" else "Scheduled Workflow"
        
        # 3. Build Safe HTML Link Anchors
        workspace_link = f"<a href='{base_url}/?o={workspace_id}' style='color: #1a73e8; text-decoration: none;'>{browser_host} [{workspace_id}]</a>"
        
        if job_id and job_run_id:
            job_link = f"<a href='{base_url}/#job/{job_id}/run/{job_run_id}' style='color: #1a73e8; text-decoration: none;'>{job_name} [{job_id}]</a>"
        elif job_id:
            job_link = f"<a href='{base_url}/#job/{job_id}' style='color: #1a73e8; text-decoration: none;'>{job_name} [{job_id}]</a>"
        else:
            job_link = "Interactive Notebook (Not a Job Run)"

        return {
            "Workspace": workspace_link,
            "Job": job_link,
            "Job Run": str(job_run_id) if job_run_id else "N/A",
            "Status": "Succeeded",
            "Launched": launched_by
        }
    except Exception as e:
        print(f"Error fetching notebook run details: {e}")
        return {
            "Workspace": "Local Development (No Link)",
            "Job": "Manual Run (No Link)",
            "Job Run": "Local Testing",
            "Status": "Succeeded",
            "Launched": "Manually"
        }

# Generate the updated dictionary format
run_details_dict = get_notebook_run_links()

In [0]:
run_details_dict